# 🏏 SupeRR Selector Season 2: Phase 1 — Data Anatomy & Deep EDA
## Mentorship & Learning: Deconstructing Event-Level Cricket Data

### What You Will Learn in This Phase:
1. **Understanding Relational Schemas in Sports Analytics**: How delivery-level events map to match context, players, and tournaments.
2. **Catching Real-World Data Anomalies**: Why raw data is never clean (e.g., categorical extra IDs disguised as run counts).
3. **Event Aggregation Mechanics**: Reconstructing cumulative match states (runs, wickets, legal balls) from ball-by-ball logs.
4. **Setting Up Feature Pipelines**: Preparing clean inputs for the Pressure & Clutch Machine Learning models.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for publication-grade sports analytics
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

--- 
### Step 1: Ingesting the 6 Core Tables
In sports analytics, event-level data (ball-by-ball) is supported by dimension tables.
- `competitions.csv`: Defines tournament level (IPL vs T20I bilateral vs World Cups).
- `matches.csv`: Match-level metadata (teams, toss, venue, knockouts, DLS revised targets).
- `teams.csv` & `playing_eleven.csv`: Lineups, captaincy, and impact player subs.
- `competition_team_player_mapping.csv`: The master player registry.
- `ball_by_ball.csv`: The fact table containing individual delivery telemetry.

In [ ]:
data_dir = '../data/raw/extracted/SupeRR Selector Season 2 - Problem Statement - Data Pack/SupeRR Selector Season 2 - Data Pack'

df_comp = pd.read_csv(os.path.join(data_dir, 'competitions.csv'))
df_matches = pd.read_csv(os.path.join(data_dir, 'matches.csv'))
df_players = pd.read_csv(os.path.join(data_dir, 'competition_team_player_mapping.csv'))
df_eleven = pd.read_csv(os.path.join(data_dir, 'playing_eleven.csv'))
df_bbb = pd.read_csv(os.path.join(data_dir, 'ball_by_ball.csv'))

print(f"[+] Competitions Loaded: {len(df_comp)} tournaments")
print(f"[+] Matches Loaded:      {len(df_matches)} matches")
print(f"[+] Player Mappings:    {len(df_players)} player records ({df_players['player_name'].nunique()} unique players)")
print(f"[+] Deliveries (BBB):    {len(df_bbb):,} ball events")

--- 
### Step 2: The Data Anatomy Lesson — Spotting the Hidden Enums!
When you first open `ball_by_ball.csv`, you see a column named `extras1_runs`.
A common novice mistake in ML is treating every column containing `'runs'` as numeric runs.

Let's see what `extras1_runs` actually contains:

In [ ]:
# Demonstrating why domain auditing is vital in sports data science
extras_inspection = df_bbb.groupby(['extras_type', 'extras1_runs'], dropna=False).size().reset_index(name='delivery_count')
extras_inspection

**Insight Revealed:**
`extras1_runs` is **not** runs! It is an **enumeration ID** representing the delivery type:
- `1` = Overstep (No-ball)
- `2` = Wide
- `3` = Leg byes
- `4` = Byes
- `5` = Other / Penalty
- `6` = Fair / Normal delivery (`extras_type = NaN`)

If an analyst blindly sums `extras1_runs`, they would add 6 runs on every single dot ball! This proves why **exploratory domain sanity checks** are non-negotiable.

--- 
### Step 3: Reconstructing Cricket Physics (Legal Balls, Runs, and Extras)
To calculate Run Rates and Bowling Economy accurately, we must differentiate:
1. **Batter Runs**: Scored strictly off the bat (`runs` column).
2. **Bowler Conceded Runs**: Batter runs + Wides + No-balls (Byes & Legbyes are unearned fielding extras).
3. **Team Total Runs**: Batter runs + All extras.
4. **Legal Balls**: Deliveries that count towards the 120-ball quota (all deliveries *except* Wides and No-balls).

In [ ]:
et = df_bbb['extras_type'].values
ex2 = df_bbb['extras_2_runs'].fillna(0).values

is_wide = (et == 'wide')
is_nb = (et == 'Overstep')
is_bye_lb = np.isin(et, ['byes', 'legbyes', 'other'])

# 1. Extras for the batting team
team_extras = np.zeros(len(df_bbb))
team_extras[is_wide] = 1 + ex2[is_wide]
team_extras[is_nb] = 1 + ex2[is_nb]
team_extras[is_bye_lb] = np.where(ex2[is_bye_lb] > 0, ex2[is_bye_lb], 1)

df_bbb['team_runs_on_ball'] = df_bbb['runs'] + team_extras

# 2. Runs charged to bowler
bowler_extras = np.zeros(len(df_bbb))
bowler_extras[is_wide] = 1 + ex2[is_wide]
bowler_extras[is_nb] = 1 + ex2[is_nb]
df_bbb['bowler_runs_on_ball'] = df_bbb['runs'] + bowler_extras

# 3. Legal delivery flag
df_bbb['is_legal_delivery'] = ~np.isin(et, ['wide', 'Overstep'])

print(f"Total team runs scored across dataset:   {df_bbb['team_runs_on_ball'].sum():,}")
print(f"Total runs charged to bowlers:           {df_bbb['bowler_runs_on_ball'].sum():,}")
print(f"Legal deliveries bowled:                 {df_bbb['is_legal_delivery'].sum():,} / {len(df_bbb):,} total balls")

--- 
### Step 4: Distribution Analysis — What Does T20 Cricket Look Like in 2022–2026?
Let's visualize the breakdown of outcomes per ball: dot balls, singles, boundaries, and wickets.

In [ ]:
outcome_counts = {
    'Dot Ball (0 runs)': (df_bbb['runs'] == 0) & (df_bbb['is_wicket'] == False) & (df_bbb['team_runs_on_ball'] == 0),
    'Singles & Twos': df_bbb['runs'].isin([1, 2, 3]),
    'Fours (4)': df_bbb['is_four'] == True,
    'Sixes (6)': df_bbb['is_six'] == True,
    'Wickets': df_bbb['is_wicket'] == True
}

summary_data = {label: mask.sum() for label, mask in outcome_counts.items()}
summary_df = pd.DataFrame(list(summary_data.items()), columns=['Outcome', 'Count'])
summary_df['Percentage'] = (summary_df['Count'] / len(df_bbb) * 100).round(2)

plt.figure(figsize=(10, 5))
bars = plt.bar(summary_df['Outcome'], summary_df['Percentage'], color=['#34495e', '#3498db', '#2ecc71', '#e74c3c', '#9b59b6'])
plt.title('Distribution of Delivery Outcomes in Modern T20 (2022–2026)', fontsize=14, fontweight='bold')
plt.ylabel('Percentage of Total Deliveries (%)')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.8, f"{yval:.1f}%", ha='center', va='bottom', fontweight='bold')
plt.ylim(0, 45)
plt.tight_layout()
plt.show()

### Key Takeaways from Phase 1:
1. **Dot Ball Dominance (~35-38%)**: Even in aggressive T20 cricket, over one-third of all balls are dots. The ability of a batter to rotate strike or a bowler to create dots is foundational.
2. **Boundary Sensitivity (~18%)**: Fours and sixes constitute under 20% of balls, but provide over 60% of total team runs. Clutch batters distinguish themselves by boundary execution under pressure.
3. **Innings Granularity**: We have 105k 1st innings deliveries, 95k 2nd innings deliveries, and 38 Super Over deliveries.

Now that the data schema and anatomy are verified, we are ready for **Phase 2: Mathematical Ideation — Defining Pressure from First Principles**.